# Entity ***Tokens***
+ Layer **silver**
+ Priority **1**
---
### Imports

In [0]:
%run ../common/medallion_functions

### Parameters

In [0]:
layer = dbutils.widgets.get("layer")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")

### Execution

In [0]:
bz_tokens_df = spark.read.table(f"uniswap.bronze.{entity_name}")

In [0]:
sv_tokens_df = spark.sql(f"""
    WITH cte_deduplicate_bz AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY id
                ORDER BY _load_timestamp_bronze DESC
            ) AS rn
        FROM {{df}}
    )
    SELECT
        id AS pk_token_id,
        symbol,
        name,
        CAST(decimals AS INT) AS decimals,
        totalSupply AS v3_held_total_supply,
        CAST(derivedETH AS DOUBLE) AS derived_price_ETH,
        ROUND(
            CAST(feesUSD AS DOUBLE),
            2
        ) AS fees_USD,
        ROUND(
            CAST(totalValueLockedUSD AS DOUBLE),
            2
        ) AS total_value_locked_USD,
        CAST(totalValueLocked AS DOUBLE) AS total_value_locked_token,
        ROUND(
            CAST(volumeUSD AS DOUBLE),
            2
        ) AS volume_USD,
        CAST(volume AS DOUBLE) AS volume_token,
        CAST(txCount AS INT) AS tx_count,
        CAST(poolCount AS INT) AS pool_count,
        CURRENT_TIMESTAMP() AS _load_timestamp_silver
    FROM cte_deduplicate_bz
    WHERE rn = 1
""", df = bz_tokens_df)

### Save & exit

In [0]:
load_result = load_entity(sv_tokens_df,
                        entity_name,
                        load_pattern,
                        layer
                        )

In [0]:
dbutils.notebook.exit(load_result)